# MCP Tutorial: Greek Search, Result Interpretation, and CTS Navigation

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction and place in the tutorial series</a>
* <a href="#architecture">2 - One MCP interface over two upstream services</a>
* <a href="#setup">3 - Install dependencies and load the MCP server</a>
* <a href="#helpers">4 - Helpers for MCP results and search summaries</a>
* <a href="#schemas">5 - Inspect the relevant tool schemas</a>
* <a href="#discover">6 - Discover Homer and the CTS Iliad edition</a>
* <a href="#greek-input">7 - Understand Unicode Greek and Beta Code input</a>
* <a href="#unicode-beta">8 - Compare Unicode and Beta Code searches</a>
* <a href="#form-lemma">9 - Compare surface-form and lemma search</a>
* <a href="#phrase-search">10 - Search for an exact Greek phrase</a>
* <a href="#scope">11 - Scope search by author or work</a>
* <a href="#result-anatomy">12 - Read a Scaife search result</a>
* <a href="#urn-mismatch">13 - Demonstrate the Scaife and CTS URN mismatch</a>
* <a href="#translate-hit">14 - Translate a Scaife hit into a CTS passage URN</a>
* <a href="#references">15 - Inspect valid CTS references</a>
* <a href="#navigation">16 - Navigate with the MCP fallback tool</a>
* <a href="#neighborhood">17 - Retrieve the previous, current, and next passages</a>
* <a href="#workflow-guide">18 - End-to-end workflow and tool-selection guide</a>
* <a href="#troubleshooting">19 - Troubleshooting and research cautions</a>
* <a href="#next-steps">20 - Continue learning</a>
* <a href="#sources">21 - Sources</a>
* <a href="#required-libraries">22 - Required libraries</a>
* <a href="#notebook-version">23 - Notebook version</a>

## 1 - Introduction and place in the tutorial series <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook is the fourth step in the introductory sequence:

- notebook `01_` exposed the plain Perseus CTS protocol;
- notebook `02_` used plain CTS and plain Scaife HTTP services to explain different kinds of search;
- notebook `03_` introduced MCP clients, tools, schemas, author discovery, and passage retrieval;
- this notebook combines those ideas into an **MCP-driven Greek search and navigation workflow**.

Unlike notebook `02_`, every research operation here is performed through MCP tools. The notebook does not call the CTS or Scaife URLs directly.

The guiding question is: **How can a user search Greek text through MCP, understand the returned Scaife URN, and safely continue reading through Perseus CTS?**

You will learn to:

- enter Greek queries as Unicode or common Beta Code;
- distinguish surface-form, lemma, and exact-phrase search;
- restrict results to Homer or the *Iliad*;
- read the important fields in a search response;
- recognize that Scaife and CTS may use different edition URNs;
- preserve the work and citation while switching to a CTS-advertised edition;
- retrieve previous and next passages with the server's navigation fallback.

## 2 - One MCP interface over two upstream services <a class="anchor" id="architecture"></a>
##### [Back to ToC](#TOC)

The local `perseus` MCP server provides one tool catalog, but the tools in this workflow use two different upstream systems:

| Research task | MCP tools used here | Upstream source |
|---|---|---|
| Discover Homer, the Iliad, and a CTS edition | `find_author_names`, `get_author_resources` | Perseus CTS inventory |
| Search words, lemmas, and phrases | `search_perseus` | Scaife search index |
| Inspect valid citations | `get_valid_references_json`, `count_valid_references` | Perseus CTS references |
| Move to neighboring citations | `get_prev_next_urn` | CTS navigation, with a local valid-reference fallback |
| Retrieve readable passage text | `get_passage_plaintext` | Perseus CTS passage XML, converted locally |

The conceptual flow is:

```text
Greek query
  → search_perseus
    → Scaife search result and Scaife edition URN
      → keep the common work URN and citation
        → combine them with a CTS edition discovered through MCP
          → CTS navigation and passage tools
```

The MCP server hides HTTP request details, normalizes Greek input, shapes selected responses, caches stable CTS metadata, and supplies navigation fallbacks. It does not erase the distinction between the underlying CTS and Scaife resources.

## 3 - Install dependencies and load the MCP server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The setup follows notebook `03_`: locate the repository root, configure a consistent cache directory, import and reload `perseus_mcp.server`, and obtain the registered FastMCP object.

Set `PERSEUS_MCP_INSTALL_SOURCE` in the install cell to `"repo"` for editable development-branch work or `"pypi"` for the published package.

The installation cell installs Perseus MCP into the active Jupyter kernel. It defaults to the local repository in editable mode for development-branch work; change the switch to `"pypi"` for the published package. For regular development, installing once with `pip install -e .` or `uv sync` from the repository root is equivalent.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *package_target,
        "python-dotenv>=1.0.0",
    ]
)

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
from pathlib import Path
import html
import importlib
import json
import os
import re
import sys
import xml.etree.ElementTree as ET

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

PERSEUS_MCP_INSTALL_SOURCE = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if PERSEUS_MCP_INSTALL_SOURCE not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

if PERSEUS_MCP_INSTALL_SOURCE == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    SRC_DIR = REPO_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
elif REPO_ROOT is not None:
    SRC_DIR = REPO_ROOT / "src"
    src_dir_resolved = SRC_DIR.resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == src_dir_resolved
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if "load_dotenv" in globals():
    if REPO_ROOT is not None:
        load_dotenv(REPO_ROOT / ".env", override=False)
    else:
        load_dotenv(override=False)

CACHE_ROOT = REPO_ROOT if REPO_ROOT is not None else START
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(CACHE_ROOT / ".cache" / "perseus-mcp"),
)

for module_name in [
    name
    for name in list(sys.modules)
    if name == "perseus_mcp" or name.startswith("perseus_mcp.")
]:
    del sys.modules[module_name]

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Install source: {PERSEUS_MCP_INSTALL_SOURCE}")
print(f"Repository root: {REPO_ROOT if REPO_ROOT is not None else 'not found'}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

## 4 - Helpers for MCP results and search summaries <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

The server returns text content blocks. Search and discovery tools serialize JSON into those blocks, while navigation tools may return XML and passage tools may return plaintext.

The helpers below:

- extract text from a FastMCP result;
- parse JSON-returning tools;
- call tools with an existing session;
- remove `<em>` highlighting from snippets;
- reduce a large Scaife result object to fields a researcher usually needs first.

In [3]:
TAG_RE = re.compile(r"<[^>]+>")


def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


def tool_json(result):
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_json(result)


def clean_snippet(content):
    joined = " | ".join(content or [])
    return " ".join(html.unescape(TAG_RE.sub("", joined)).split())


def compact_search_rows(data, limit=5):
    rows = []
    for result in data.get("results", [])[:limit]:
        passage = result.get("passage", {})
        text = passage.get("text", {})
        ancestors = text.get("ancestors", [])
        rows.append(
            {
                "passage_urn": passage.get("urn"),
                "edition_urn": text.get("urn"),
                "author": ancestors[0].get("label") if len(ancestors) > 0 else None,
                "work": ancestors[1].get("label") if len(ancestors) > 1 else None,
                "reference": passage.get("refs", {}).get("start", {}).get("human_reference"),
                "snippet": clean_snippet(result.get("content")),
            }
        )
    return rows


def print_search_summary(label, data, limit=3):
    print(f"\n{label}")
    print(
        f"normalized query={data.get('q')!r}; "
        f"kind={data.get('kind')!r}; "
        f"total_count={data.get('total_count')}"
    )
    print(json.dumps(compact_search_rows(data, limit), ensure_ascii=False, indent=2))


def local_name(tag):
    return tag.rsplit("}", 1)[-1]

## 5 - Inspect the relevant tool schemas <a class="anchor" id="schemas"></a>
##### [Back to ToC](#TOC)

Before calling unfamiliar tools, inspect their descriptions and input schemas. This makes the accepted options visible and reduces trial-and-error.

For `search_perseus`, pay particular attention to:

- `query` — the text to search;
- `language` — controls Greek query normalization;
- `query_format` — `auto`, `unicode`, or `betacode`;
- `search_kind` — `form` or `lemma`;
- `preserve_operators` — keep quotes, `-`, `|`, `*`, or `~` intact;
- `author`, `text_group`, and `work` — different ways to narrow the corpus;
- `page_num` and `result_format` — pagination and result grouping.

In [4]:
async with Client(mcp) as client:
    tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in tools}
for name in [
    "search_perseus",
    "get_author_resources",
    "get_valid_references_json",
    "get_prev_next_urn",
    "get_passage_plaintext",
]:
    tool = tool_by_name[name]
    print(f"\n{name}")
    print(tool.description)
    print(json.dumps(tool.inputSchema, ensure_ascii=False, indent=2))


search_perseus
Search Perseus texts via Scaife API.

For Greek searches, `query` may be Unicode Greek or Beta Code.  The default
`query_format="auto"` detects explicit Beta Code marks such as `=`, `/`,
`(`, `)`, and `*`, and also accepts short unaccented Beta Code queries such
as `logos`.  Set `query_format="betacode"` to force conversion or
`query_format="unicode"` to preserve ASCII text in Greek searches.
The `language` value determines whether Greek query normalization is applied;
it is not sent to Scaife as a corpus language filter.
Optional `author` resolves a CTS author/textgroup name or URN, then locally
filters the current Scaife result page to matching CTS URN prefixes.
`search_kind` may be "form" or "lemma". Set `preserve_operators=True` for
Scaife operator queries such as quoted phrases, `-`, `|`, `*`, or `~`.
Optional `page_num`, `text_group`, `work`, and `result_format` are passed
to Scaife's library search endpoint. When `author` resolves to exactly one
CTS textgroup and

## 6 - Discover Homer and the CTS Iliad edition <a class="anchor" id="discover"></a>
##### [Back to ToC](#TOC)

Search results will come from Scaife, but later navigation will use Perseus CTS. We therefore discover the current CTS resource before searching.

The workflow uses `find_author_names` to disambiguate `Homer` from names such as `Homeric Hymns`, then passes the exact textgroup URN to `get_author_resources`. Finally, it selects the *Iliad* and one advertised Greek edition.

Do not assume that a suffix such as `perseus-grc1` is permanent or that it will match Scaife's edition suffix.

In [5]:
async with Client(mcp) as client:
    author_matches = await call_json(
        client,
        "find_author_names",
        {"query": "Hom", "language": "greek", "limit": 10},
    )

homer_match = next(
    (author for author in author_matches["authors"] if "Homer" in author["names"]),
    None,
)
if homer_match is None:
    raise RuntimeError("The current CTS inventory returned no exact Homer match.")

HOMER_TEXTGROUP = homer_match["urn"]

async with Client(mcp) as client:
    homer_resources = await call_json(
        client,
        "get_author_resources",
        {"author": HOMER_TEXTGROUP, "language": "greek"},
    )

if not homer_resources["authors"]:
    raise RuntimeError("No Homer resources were returned.")

homer_author = homer_resources["authors"][0]
iliad_work = next(
    (work for work in homer_author["works"] if "Iliad" in work["titles"]),
    None,
)
if iliad_work is None:
    raise RuntimeError("No work titled Iliad was returned for Homer.")

greek_editions = [
    edition
    for edition in iliad_work["editions"]
    if edition.get("language") == "grc"
    or "-grc" in (edition.get("urn") or "")
    or ".perseus-grc" in (edition.get("urn") or "")
]
if not greek_editions:
    raise RuntimeError("No Greek Iliad edition is advertised by CTS.")

ILIAD_WORK = iliad_work["urn"]
CTS_ILIAD_EDITION = greek_editions[0]["urn"]

print(
    json.dumps(
        {
            "author_textgroup": HOMER_TEXTGROUP,
            "work_urn": ILIAD_WORK,
            "work_titles": iliad_work["titles"],
            "available_greek_cts_editions": greek_editions,
            "selected_cts_edition": CTS_ILIAD_EDITION,
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "author_textgroup": "urn:cts:greekLit:tlg0012",
  "work_urn": "urn:cts:greekLit:tlg0012.tlg001",
  "work_titles": [
    "Iliad"
  ],
  "available_greek_cts_editions": [
    {
      "type": "edition",
      "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "label": "Iliad",
      "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
    }
  ],
  "selected_cts_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
}


## 7 - Understand Unicode Greek and Beta Code input <a class="anchor" id="greek-input"></a>
##### [Back to ToC](#TOC)

Greek search input can arrive in two common forms:

| Input style | Example | Appropriate `query_format` |
|---|---|---|
| Polytonic Unicode | `μῆνιν` | `unicode` or `auto` |
| Beta Code | `mh=nin` | `betacode` or `auto` |
| Ambiguous unaccented ASCII | `logos` | Prefer explicit `betacode` if Greek is intended |

The MCP server normalizes Greek queries to composed Unicode before sending them to Scaife. Explicit Beta Code markers include characters such as `=`, `/`, `(`, `)`, and `*`.

Important limits:

- the converter supports common Beta Code input; it is not presented as a complete interchange validator;
- `language="greek"` controls normalization but is not itself a Scaife corpus-language filter;
- if a query contains search operators, use Unicode plus `preserve_operators=True` so operator characters are not interpreted as Beta Code marks;
- the normalized query can be inspected in the returned JSON field `q`.

## 8 - Compare Unicode and Beta Code searches <a class="anchor" id="unicode-beta"></a>
##### [Back to ToC](#TOC)

The next calls ask the same work-scoped surface-form question twice: once with Unicode `μῆνιν` and once with Beta Code `mh=nin`.

Both calls go through `search_perseus`. If normalization succeeds, the response's `q` field and the first result URNs should agree. Comparing returned identifiers is more reliable than comparing only snippets.

In [6]:
async with Client(mcp) as client:
    unicode_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "work": ILIAD_WORK,
        },
    )
    betacode_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "mh=nin",
            "language": "greek",
            "query_format": "betacode",
            "search_kind": "form",
            "work": ILIAD_WORK,
        },
    )

unicode_urns = [row["passage_urn"] for row in compact_search_rows(unicode_search, 10)]
betacode_urns = [row["passage_urn"] for row in compact_search_rows(betacode_search, 10)]

print(
    json.dumps(
        {
            "unicode_input": "μῆνιν",
            "unicode_normalized_q": unicode_search.get("q"),
            "betacode_input": "mh=nin",
            "betacode_normalized_q": betacode_search.get("q"),
            "unicode_total": unicode_search.get("total_count"),
            "betacode_total": betacode_search.get("total_count"),
            "same_first_page_urns": unicode_urns == betacode_urns,
        },
        ensure_ascii=False,
        indent=2,
    )
)
print_search_summary("Unicode search results", unicode_search, limit=3)

{
  "unicode_input": "μῆνιν",
  "unicode_normalized_q": "μῆνιν",
  "betacode_input": "mh=nin",
  "betacode_normalized_q": "μῆνιν",
  "unicode_total": 9,
  "betacode_total": 9,
  "same_first_page_urns": true
}

Unicode search results
normalized query='μῆνιν'; kind='form'; total_count=9
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 1 Line 75",
    "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 5 Line 444",
    "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:16.711",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",


## 9 - Compare surface-form and lemma search <a class="anchor" id="form-lemma"></a>
##### [Back to ToC](#TOC)

A **surface-form search** asks for a visible spelling in the text. A **lemma search** asks for tokens indexed under a dictionary headword.

For this example:

- form query `μῆνιν` searches the accusative form as written;
- lemma query `μῆνις` searches the lexical headword and may return inflected forms such as `μῆνιν`.

Do not submit an inflected form such as `μῆνιν` and assume it is a valid lemma query. Also avoid treating `total_count` as a definitive corpus frequency without understanding Scaife's indexing and result grouping.

In [7]:
async with Client(mcp) as client:
    lemma_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνις",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "work": ILIAD_WORK,
        },
    )

print_search_summary("Surface form: μῆνιν", unicode_search, limit=3)
print_search_summary("Lemma: μῆνις", lemma_search, limit=3)
print(
    json.dumps(
        {
            "form_total_count": unicode_search.get("total_count"),
            "lemma_total_count": lemma_search.get("total_count"),
            "lemma_first_snippets": [
                row["snippet"] for row in compact_search_rows(lemma_search, 3)
            ],
        },
        ensure_ascii=False,
        indent=2,
    )
)


Surface form: μῆνιν
normalized query='μῆνιν'; kind='form'; total_count=9
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 1 Line 75",
    "snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 5 Line 444",
    "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
  },
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:16.711",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 16 Line 711",
    "snippet": "μῆνιν ἀλευάμενος ἑκατηβόλου Ἀπόλλωνος."
  }
]

Lemma: μῆνις
normalized query='μῆνις'; kind='lemma'; total_count=12


## 10 - Search for an exact Greek phrase <a class="anchor" id="phrase-search"></a>
##### [Back to ToC](#TOC)

Scaife supports quoted phrase syntax. Because quotation marks are search operators, this call uses:

- Unicode Greek input;
- `query_format="unicode"`;
- `preserve_operators=True`;
- the *Iliad* work scope.

This combination keeps the phrase quotes intact and avoids confusing operator characters with Beta Code notation.

In [8]:
async with Client(mcp) as client:
    phrase_search = await call_json(
        client,
        "search_perseus",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "preserve_operators": True,
            "work": ILIAD_WORK,
        },
    )

print_search_summary("Exact phrase in the Iliad", phrase_search, limit=5)


Exact phrase in the Iliad
normalized query='"μῆνιν ἄειδε"'; kind='form'; total_count=1
[
  {
    "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
    "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "author": "Homer",
    "work": "Iliad",
    "reference": "Book 1 Line 1",
    "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"
  }
]


## 11 - Scope search by author or work <a class="anchor" id="scope"></a>
##### [Back to ToC](#TOC)

`search_perseus` supports three related scoping approaches:

| Argument | Typical use |
|---|---|
| `author` | Convenience input using an author name or textgroup URN; the server resolves it through CTS |
| `text_group` | Explicit Scaife textgroup filter when the CTS URN is already known |
| `work` | Explicit Scaife work filter, such as the *Iliad* rather than all Homeric works |

When `author` resolves to exactly one textgroup and no explicit work is supplied, the server sends the resolved textgroup to Scaife as a server-side filter and adds `author_scope` metadata to the result.

The counts below answer progressively narrower questions: anywhere in the library, anywhere under Homer, and specifically in the *Iliad*.

In [9]:
async with Client(mcp) as client:
    library_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
        },
    )
    homer_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "author": HOMER_TEXTGROUP,
        },
    )

scope_summary = {
    "unscoped_library_total": library_search.get("total_count"),
    "homer_textgroup_total": homer_search.get("total_count"),
    "iliad_work_total": unicode_search.get("total_count"),
    "resolved_author_scope": homer_search.get("author_scope"),
}
print(json.dumps(scope_summary, ensure_ascii=False, indent=2))

{
  "unscoped_library_total": 314,
  "homer_textgroup_total": 12,
  "iliad_work_total": 9,
  "resolved_author_scope": {
    "query": "urn:cts:greekLit:tlg0012",
    "match_count": 1,
    "text_group": "urn:cts:greekLit:tlg0012",
    "note": "Author scope was sent to Scaife as a server-side text_group filter."
  }
}


## 12 - Read a Scaife search result <a class="anchor" id="result-anatomy"></a>
##### [Back to ToC](#TOC)

A `search_perseus` result contains more than a snippet. Important fields include:

| Field | Meaning |
|---|---|
| `passage.urn` | Edition-specific Scaife passage URN |
| `passage.text.urn` | Scaife edition/text URN |
| `passage.text.ancestors` | Textgroup/author and work metadata |
| `passage.refs.start` | Machine and human-readable citation information |
| `content` | One or more snippets, often with `<em>` around matches |
| `total_count` | Number reported by the current indexed search |
| `page` | Pagination metadata |

Keep the passage URN beside the snippet. A text fragment without its source identifier is much harder to verify later.

In [10]:
if not unicode_search.get("results"):
    raise RuntimeError("The work-scoped Iliad search returned no results.")

first_hit = unicode_search["results"][0]
first_passage = first_hit["passage"]
first_text = first_passage["text"]

result_anatomy = {
    "passage_urn": first_passage.get("urn"),
    "edition_urn": first_text.get("urn"),
    "ancestors": first_text.get("ancestors"),
    "reference": first_passage.get("refs", {}).get("start"),
    "raw_content": first_hit.get("content"),
    "clean_snippet": clean_snippet(first_hit.get("content")),
    "page": unicode_search.get("page"),
}
print(json.dumps(result_anatomy, ensure_ascii=False, indent=2))

{
  "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
  "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "ancestors": [
    {
      "url": "/library/urn:cts:greekLit:tlg0012/",
      "json_url": "/library/urn:cts:greekLit:tlg0012/json/",
      "text_url": "/library/passage/urn:cts:greekLit:tlg0012/text/",
      "urn": "urn:cts:greekLit:tlg0012",
      "label": "Homer"
    },
    {
      "url": "/library/urn:cts:greekLit:tlg0012.tlg001/",
      "json_url": "/library/urn:cts:greekLit:tlg0012.tlg001/json/",
      "text_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001/text/",
      "urn": "urn:cts:greekLit:tlg0012.tlg001",
      "label": "Iliad"
    }
  ],
  "reference": {
    "reference": "1.75",
    "human_reference": "Book 1 Line 75"
  },
  "raw_content": [
    "<em>μῆνιν</em> Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·"
  ],
  "clean_snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·",
  "page": {
    "number": 1,
    "start_index": 1,
    "end_index": 9,
   

## 13 - Demonstrate the Scaife and CTS URN mismatch <a class="anchor" id="urn-mismatch"></a>
##### [Back to ToC](#TOC)

The discovery tool selected an edition from the **Perseus CTS inventory**. The search tool returned a passage from the **Scaife index**. These services can use different edition URNs for the same author and work.

In the current live data, the difference is commonly:

- CTS: `urn:cts:greekLit:tlg0012.tlg001.perseus-grc1`
- Scaife: `urn:cts:greekLit:tlg0012.tlg001.perseus-grc2`

The common work URN (`urn:cts:greekLit:tlg0012.tlg001`) and citation can still provide a bridge, but the edition identifiers must not be treated as interchangeable. Editions may differ in punctuation, orthography, markup, or substantive readings.

In [11]:
SCAIFE_PASSAGE_URN = first_passage["urn"]
SCAIFE_ILIAD_EDITION = first_text["urn"]
scaife_work_urn = SCAIFE_ILIAD_EDITION.rsplit(".", 1)[0]
SEARCH_HIT_CITATION = SCAIFE_PASSAGE_URN.rpartition(":")[2]

print(
    json.dumps(
        {
            "same_work_urn": scaife_work_urn == ILIAD_WORK,
            "work_urn": ILIAD_WORK,
            "cts_edition": CTS_ILIAD_EDITION,
            "scaife_edition": SCAIFE_ILIAD_EDITION,
            "same_edition_urn": CTS_ILIAD_EDITION == SCAIFE_ILIAD_EDITION,
            "search_hit_citation": SEARCH_HIT_CITATION,
            "scaife_passage_urn": SCAIFE_PASSAGE_URN,
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "same_work_urn": true,
  "work_urn": "urn:cts:greekLit:tlg0012.tlg001",
  "cts_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "scaife_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "same_edition_urn": false,
  "search_hit_citation": "1.75",
  "scaife_passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75"
}


## 14 - Translate a Scaife hit into a CTS passage URN <a class="anchor" id="translate-hit"></a>
##### [Back to ToC](#TOC)

To continue with CTS-backed tools, preserve the citation from the Scaife hit but replace the Scaife edition prefix with the edition discovered from CTS:

```text
Scaife edition + citation
urn:cts:...perseus-grc2:1.75

CTS edition + same citation
urn:cts:...perseus-grc1:1.75
```

This is a controlled mapping, not a claim that the editions are identical. The next sections verify that the translated URN is valid in the current CTS inventory and retrieve its neighborhood.

In [12]:
CTS_PASSAGE_URN = f"{CTS_ILIAD_EDITION}:{SEARCH_HIT_CITATION}"

print(
    json.dumps(
        {
            "source_scaife_passage": SCAIFE_PASSAGE_URN,
            "shared_work": ILIAD_WORK,
            "shared_citation": SEARCH_HIT_CITATION,
            "target_cts_passage": CTS_PASSAGE_URN,
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "source_scaife_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
  "shared_work": "urn:cts:greekLit:tlg0012.tlg001",
  "shared_citation": "1.75",
  "target_cts_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75"
}


## 15 - Inspect valid CTS references <a class="anchor" id="references"></a>
##### [Back to ToC](#TOC)

Before navigation, inspect the CTS edition's citation inventory. The paged JSON helper is preferable to printing the complete raw `GetValidReff` XML.

`count_valid_references` reports the total, while `get_valid_references_json` returns a small page. The page below begins at offset zero; it demonstrates response shape rather than locating `1.75` by offset.

In [13]:
async with Client(mcp) as client:
    reference_count = await call_json(
        client,
        "count_valid_references",
        {"urn": CTS_ILIAD_EDITION},
    )
    first_reference_page = await call_json(
        client,
        "get_valid_references_json",
        {"urn": CTS_ILIAD_EDITION, "limit": 5, "offset": 0},
    )

print("Reference count:")
print(json.dumps(reference_count, ensure_ascii=False, indent=2))
print("\nFirst page:")
print(json.dumps(first_reference_page, ensure_ascii=False, indent=2))

Reference count:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": null,
  "total_count": 14956
}

First page:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": null,
  "total_count": 14956,
  "offset": 0,
  "limit": 5,
  "returned_count": 5,
  "has_more": true,
  "references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5"
  ]
}


## 16 - Navigate with the MCP fallback tool <a class="anchor" id="navigation"></a>
##### [Back to ToC](#TOC)

`get_prev_next_urn` first asks the live CTS service for neighboring URNs. If that route returns malformed HTML instead of the expected XML, the local server loads the ordered valid references and creates a well-formed navigation response.

This fallback is one of the practical benefits of using the MCP tool rather than calling the upstream navigation endpoint directly.

The tool returns XML, so the notebook parses `previous` and `next` elements into a small Python dictionary.

In [14]:
async with Client(mcp) as client:
    navigation_xml = await call_text(
        client,
        "get_prev_next_urn",
        {"urn": CTS_PASSAGE_URN},
    )

navigation_root = ET.fromstring(navigation_xml)
navigation = {"previous": None, "current": CTS_PASSAGE_URN, "next": None}

for element in navigation_root.iter():
    name = local_name(element.tag).casefold()
    if name not in {"previous", "next"}:
        continue
    urn_element = next(
        (child for child in element.iter() if local_name(child.tag).casefold() == "urn"),
        None,
    )
    if urn_element is not None and urn_element.text:
        navigation[name] = urn_element.text.strip()

print("Raw navigation XML:")
print(navigation_xml)
print("\nParsed navigation:")
print(json.dumps(navigation, ensure_ascii=False, indent=2))

Raw navigation XML:
<GetPrevNextUrn><request><requestName>GetPrevNextUrn</requestName><requestUrn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75</requestUrn></request><reply><previous><urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.74</urn></previous><next><urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.76</urn></next></reply></GetPrevNextUrn>

Parsed navigation:
{
  "previous": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.74",
  "current": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75",
  "next": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.76"
}


## 17 - Retrieve the previous, current, and next passages <a class="anchor" id="neighborhood"></a>
##### [Back to ToC](#TOC)

Navigation URNs become useful when they are paired with text. The code calls `get_passage_plaintext` for each available neighbor and keeps the URN beside the returned passage.

The current CTS passage is also compared with the original Scaife snippet. Small differences are possible because the search and passage services may refer to different editions.

In [16]:
passage_neighborhood = []

async with Client(mcp) as client:
    for position in ["previous", "current", "next"]:
        urn = navigation[position]
        if urn is None:
            continue
        text = await call_text(
            client,
            "get_passage_plaintext",
            {"urn": urn},
        )
        passage_neighborhood.append(
            {"position": position, "urn": urn, "text": text}
        )

for passage in passage_neighborhood:
    print(f"\n{passage['position'].upper()} — {passage['urn']}")
    print(passage["text"])

current_cts_text = next(
    passage["text"]
    for passage in passage_neighborhood
    if passage["position"] == "current"
)
print("\nSearch/passage comparison:")
print(
    json.dumps(
        {
            "scaife_snippet": clean_snippet(first_hit.get("content")),
            "cts_passage_text": current_cts_text,
            "identical_strings": clean_snippet(first_hit.get("content")) == current_cts_text,
        },
        ensure_ascii=False,
        indent=2,
    )
)


PREVIOUS — urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.74
ὦ Ἀχιλεῦ κέλεαί με Διῒ φίλε μυθήσασθαι

CURRENT — urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.75
μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος:

NEXT — urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.76
τοὶ γὰρ ἐγὼν ἐρέω: σὺ δὲ σύνθεο καί μοι ὄμοσσον

Search/passage comparison:
{
  "scaife_snippet": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·",
  "cts_passage_text": "μῆνιν Ἀπόλλωνος ἑκατηβελέταο ἄνακτος:",
  "identical_strings": false
}


## 18 - End-to-end workflow and tool-selection guide <a class="anchor" id="workflow-guide"></a>
##### [Back to ToC](#TOC)

The complete research pattern demonstrated here is:

1. **Discover identity:** `find_author_names`.
2. **Discover CTS resources:** `get_author_resources` or `get_work_resources`.
3. **Search the indexed library:** `search_perseus`.
4. **Interpret the hit:** keep author, work, edition, citation, and snippet together.
5. **Check the service boundary:** compare the Scaife edition with the discovered CTS edition.
6. **Map carefully:** preserve work and citation; substitute only the CTS edition prefix.
7. **Inspect citations:** `count_valid_references` or `get_valid_references_json`.
8. **Navigate:** `get_prev_next_urn`.
9. **Read:** `get_passage_plaintext`.

| Question | Tool and important options |
|---|---|
| Exact visible spelling? | `search_perseus(search_kind="form")` |
| Dictionary headword? | `search_perseus(search_kind="lemma")` |
| Beta Code input? | `query_format="betacode"` or carefully use `auto` |
| Quoted phrase/operators? | Unicode input with `preserve_operators=True` |
| One author? | `author=...` for CTS-assisted resolution or `text_group=...` explicitly |
| One work? | `work=...` |
| Another results page? | `page_num=...` |
| Search one chosen Scaife edition? | `search_within_text` |
| Token positions in one passage? | `get_passage_highlights` |
| Neighboring CTS passages? | `get_prev_next_urn` followed by `get_passage_plaintext` |

## 19 - Troubleshooting and research cautions <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Explanation or response |
|---|---|
| Beta Code returns an unexpected query | Set `query_format="betacode"` explicitly and inspect the response's `q` field |
| Quotes, `-`, `|`, `*`, or `~` behave strangely | Use Unicode input with `preserve_operators=True` |
| Lemma search returns zero | Submit the dictionary headword rather than an inflected surface form |
| An unscoped search returns surprising authors | Add `author`, `text_group`, or `work` scope |
| Search and CTS edition URNs differ | This is expected across Scaife and CTS; preserve work/citation and use a discovered CTS edition for CTS tools |
| The translated citation is not valid in CTS | Do not force it; inspect CTS valid references and treat the editions as non-aligning at that citation |
| Navigation upstream is malformed | Use `get_prev_next_urn`; the server can derive neighbors from valid references |
| Search counts change | Scaife is a live index; record the query, scope, options, and date |
| Cache reads fail or seem stale | Inspect `get_cache_status`; refresh or clear the metadata cache in a fresh session if appropriate |

Research cautions:

- a search snippet is evidence locator, not full context;
- a matching citation does not prove two editions have identical text;
- lemma results depend on the search index's morphological annotations;
- retain URNs and query options with exported evidence;
- verify important claims against the complete passage and, where relevant, the chosen edition.

## 20 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Good experiments within this notebook:

- compare `query_format="auto"` with explicit `unicode` and `betacode`;
- search another Homeric form and then its dictionary lemma;
- change `page_num` and observe pagination metadata;
- select an *Odyssey* work/edition through discovery and repeat the workflow;
- call `get_first_urn` for the selected CTS edition;
- use `search_within_text` with `SCAIFE_ILIAD_EDITION` rather than a library-wide work scope;
- call `get_passage_highlights` for `SCAIFE_PASSAGE_URN`.

Continue with:

- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) to inspect the complete live tool catalog;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) for operator syntax and deeper search comparisons;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) for reader search, highlights, cache controls, and Scaife-native retrieval;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) to let an LLM choose and call the same tools after the manual workflow is understood.

## 21 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook uses:

- the local MCP implementation in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- the project overview and tool documentation in the [README](../README.md);
- [FastMCP](https://github.com/jlowin/fastmcp) for the client, server, schemas, validation, and in-process transport;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS service for discovery, references, navigation, and passages;
- the [Scaife Viewer](https://scaife.perseus.org/) search service for form, lemma, phrase, and scoped search results.

Perseus CTS and Scaife are separate live services. Their inventories, edition URNs, indexed counts, ordering, and textual details may change. The notebook discovers resources at runtime and exposes the cross-service mismatch rather than hiding it.

## 22 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Runtime dependencies are:

- `fastmcp>=2.12.0` for the MCP client and server;
- `httpx>=0.27.0` for upstream HTTP requests made by the server;
- `python-dotenv>=1.0.0` for notebook-only environment-file support; it is not a core package dependency;
- Jupyter/IPython for notebook execution.

`html`, `importlib`, `json`, `os`, `pathlib`, `re`, `sys`, and `xml.etree.ElementTree` are Python standard-library modules.

Recommended project installation:

```bash
pip install -e .
```

or:

```bash
uv sync
```

## 23 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.2</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>